# Qwen branch-aware tree rerank compare (Colab)

Этот ноутбук запускает compare между:
- base scoring
- anchor rerank
- **branch-aware tree rerank**

Он использует `scripts/compare_qwen_tree_branch_rerank.py` и пишет JSON + Markdown report.


> Рекомендуемый runtime в Colab: **GPU**. Для rerank это не так критично, как для long generation, но всё равно заметно ускоряет прогон.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q torch transformers einops pytest


In [ ]:
%cd /content
import os
if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
%cd /content/ABPT
!git fetch --all
!git checkout main
!git pull --ff-only origin main


In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
DEVICE = 'cuda'
MAX_LENGTH = 256
FUTURE_WINDOW = 16
SPAN_THRESHOLD = 0.75
TOP_SPANS = 4
SEED = 7
RERANK_STRENGTH = 0.35
TREE_STRENGTH = 0.45

# '' = весь набор. Можно поставить, например: 'api_framework,quantifier'
CASE_FILTER = ''

OUTPUT_JSON = 'archive/qwen_tree_branch_rerank_compare.json'
OUTPUT_MD = 'docs/research/qwen_tree_branch_rerank_compare.md'


In [ ]:
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable,
    'scripts/compare_qwen_tree_branch_rerank.py',
    '--model', MODEL_NAME,
    '--device', DEVICE,
    '--max_length', str(MAX_LENGTH),
    '--future_window', str(FUTURE_WINDOW),
    '--span_threshold', str(SPAN_THRESHOLD),
    '--top_spans', str(TOP_SPANS),
    '--seed', str(SEED),
    '--rerank_strength', str(RERANK_STRENGTH),
    '--tree_strength', str(TREE_STRENGTH),
    '--case_filter', CASE_FILTER,
    '--output_json', OUTPUT_JSON,
    '--output_md', OUTPUT_MD,
]
print('RUNNING:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'branch compare script failed with code {result.returncode}')
assert Path(OUTPUT_JSON).exists(), f'missing output json: {OUTPUT_JSON}'
assert Path(OUTPUT_MD).exists(), f'missing output md: {OUTPUT_MD}'


In [ ]:
import json
from pathlib import Path

payload = json.loads(Path(OUTPUT_JSON).read_text(encoding='utf-8'))
summary = payload['summary']
print('case_count:', summary['case_count'])
print('base_accuracy:', summary['base_accuracy'])
print('anchor_accuracy:', summary['anchor_accuracy'])
print('branch_accuracy:', summary['branch_accuracy'])
print('branch_minus_base_accuracy:', summary['branch_minus_base_accuracy'])
print('branch_minus_anchor_accuracy:', summary['branch_minus_anchor_accuracy'])
print('branch_rescued_cases:', summary['branch_rescued_cases'])
print('branch_regressed_cases:', summary['branch_regressed_cases'])


In [ ]:
for item in payload['results']:
    print('===', item['name'], '===')
    print('base_correct:', item['base_correct'])
    print('anchor_correct:', item['anchor_correct'])
    print('branch_correct:', item['branch_correct'])
    print('base_margin:', item['base_margin'])
    print('anchor_margin:', item['anchor_margin'])
    print('branch_margin:', item['branch_margin'])
    print('preferred tree bonus:', item['preferred']['tree_bonus'])
    print('rejected tree bonus:', item['rejected']['tree_bonus'])
    print()


In [ ]:
print(Path(OUTPUT_MD).read_text(encoding='utf-8')[:12000])
